This part of the pipeline estimates the genomic divergence rate of each order using Panstripe.

### Paths and parameters

#### Pipeline input folders

In [1]:
pa.file = "03-pangenomes/all/gene_presence_absence.Rtab"
grouping.file = "02-GTDB/filtered_classification_table"
subtrees.folder = "04-core-phylogeny/subtrees"

#### Pipeline output folders

In [2]:
task_root = "08-temporal-analysis"
system(paste0('mkdir -p ', task_root), intern = TRUE)

character(0)

#### Tool pointers and parameters

In [4]:
set.seed(127)

In [5]:
library(panstripe)
library(ape)
library(ggplot2)

### Load files and metadata

#### Presence/absence files

In [ ]:
pa = read_rtab(pa.file)

In [ ]:
nrow(pa)

#### Grouping

In [ ]:
grouping = read.table(grouping.file, col.names = c('accession', 'group'))

In [ ]:
grouping

#### Phylogenies

In [ ]:
tree.files = sapply(c(unique(grouping$group), 'all'), function(x) paste('04-core-phylogeny', 'subtrees', paste0(x, '.contree'), sep = "/"))

In [ ]:
tree.files

In [ ]:
trees = lapply(tree.files, read.tree)

### Fitting genomic divergence models

using Gaussian GLMs for robustness and ease of convergence

In [ ]:
extract_subpa = function(pa, requested_group) {
    subpa = pa[grouping[grouping$group == requested_group,]$accession,]
    return(subpa)
}

In [ ]:
fit.clostridiales = panstripe(extract_subpa(pa, 'Clostridiales'), trees[['Clostridiales']], family = "gaussian")
fit.lachnospirales = panstripe(extract_subpa(pa, 'Lachnospirales'), trees[['Lachnospirales']], family = "gaussian")
fit.oscillospirales = panstripe(extract_subpa(pa, 'Oscillospirales'), trees[['Oscillospirales']], family = "gaussian")
fit.peptostreptococcales = panstripe(extract_subpa(pa, 'Peptostreptococcales'), trees[['Peptostreptococcales']], family = "gaussian")

In [ ]:
plot_residuals(fit.clostridiales)

In [ ]:
plot_residuals(fit.lachnospirales)

In [ ]:
plot_residuals(fit.oscillospirales)

In [ ]:
plot_residuals(fit.peptostreptococcales)

In [7]:
fit.clostridiales$summary

term,estimate,std.error,statistic,p.value,bootstrap CI 2.5%,bootstrap CI 97.5%
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
istip,363.3901,42.27946,8.594956,1.748500e-16,289.79,438.97
core,5140.2852,603.73876,8.514088,3.170809e-16,4095.70,6613.20
depth,649.0734,78.82909,8.233932,2.420182e-15,517.93,763.93
istip:core,2424.6369,851.75533,2.846635,4.639259e-03,129.19,4425.20
p,NA,NA,NA,NA,NA,NA
phi,NA,NA,NA,NA,NA,NA


In [8]:
fit.lachnospirales$summary

term,estimate,std.error,statistic,p.value,bootstrap CI 2.5%,bootstrap CI 97.5%
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
istip,621.02858,35.87893,17.30900415,4.833039e-56,557.24,686.69
core,5995.24742,547.81822,10.94386266,8.071787e-26,4941.60,7427.10
depth,343.42895,55.27824,6.21273330,8.987839e-10,255.34,415.76
istip:core,-31.34989,717.20984,-0.04371091,9.651475e-01,-1859.30,1674.50
p,NA,NA,NA,NA,NA,NA
phi,NA,NA,NA,NA,NA,NA


In [9]:
fit.oscillospirales$summary

term,estimate,std.error,statistic,p.value,bootstrap CI 2.5%,bootstrap CI 97.5%
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
istip,445.7951,35.63145,12.511280,1.537169e-29,386.04,509.06
core,4218.2717,412.98011,10.214225,2.113208e-21,3673.70,4725.00
depth,355.2353,63.55832,5.589124,4.868340e-08,286.70,421.75
istip:core,2189.6013,585.28856,3.741063,2.168948e-04,1155.40,3457.10
p,NA,NA,NA,NA,NA,NA
phi,NA,NA,NA,NA,NA,NA


In [10]:
fit.peptostreptococcales$summary

term,estimate,std.error,statistic,p.value,bootstrap CI 2.5%,bootstrap CI 97.5%
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
istip,537.8003,67.46863,7.9711167,1.674597e-13,417.25,692.95
core,5222.0630,684.71648,7.6266063,1.294867e-12,3992.80,7158.20
depth,174.5821,134.65725,1.2964924,1.964474e-01,-119.74,386.46
istip:core,-590.3913,952.18043,-0.6200414,5.360060e-01,-3224.40,1453.30
p,NA,NA,NA,NA,NA,NA
phi,NA,NA,NA,NA,NA,NA


In [ ]:
svg(paste(task_root, 'panstripe_cumulative_pangenome.svg', sep = "/"))
plot_pangenome_cumulative(list(Clostridiales = fit.clostridiales, 
                               Lachnospirales = fit.lachnospirales, 
                               Oscillospirales = fit.oscillospirales, 
                               Peptostreptococcales = fit.peptostreptococcales))
dev.off()

#### Statistically comparing the model fits

In [7]:
compare_pangenomes(fit.clostridiales, fit.lachnospirales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic    p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>      <dbl>               <dbl>
1 depth    -306.     102.      -2.99 0.00281                  -450.
2 istip     258.      58.0      4.44 0.00000980                156.
3 core      855.     848.       1.01 0.314                    -974.
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
               363.4                5140.3                 649.1  
          istip:core       depth:pangenome       istip:pangenome  
              2424.6                -305.6                 257.6  
      core:pangenome  istip:core:pangenome  
               855.0               -2456.0  

Degrees of Freedom: 1112 Total (i.e. Null);  1104 Residual
Null Deviance:	    727900000 
Residual Deviance: 99850000 	AIC: 15860

$data
# 

In [11]:
compare_pangenomes(fit.clostridiales, fit.oscillospirales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>   <dbl>               <dbl>
1 depth   -294.      104.      -2.83 0.00484              -427. 
2 istip     82.4      57.1      1.44 0.150                 -21.8
3 core    -922.      734.      -1.26 0.210               -2542. 
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
              363.39               5140.29                649.07  
          istip:core       depth:pangenome       istip:pangenome  
             2424.64               -293.84                 82.41  
      core:pangenome  istip:core:pangenome  
             -922.01               -235.04  

Degrees of Freedom: 742 Total (i.e. Null);  734 Residual
Null Deviance:	    4.04e+08 
Residual Deviance: 43520000 	AIC: 10270

$data
# A tibble: 742 × 5


In [12]:
compare_pangenomes(fit.clostridiales, fit.peptostreptococcales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>   <dbl>               <dbl>
1 depth   -474.      152.    -3.11   0.00193             -771.  
2 istip    174.       77.9    2.24   0.0255                 4.93
3 core      81.8     901.     0.0908 0.928              -2247.  
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
              363.39               5140.29                649.07  
          istip:core       depth:pangenome       istip:pangenome  
             2424.64               -474.49                174.41  
      core:pangenome  istip:core:pangenome  
               81.78              -3015.03  

Degrees of Freedom: 602 Total (i.e. Null);  594 Residual
Null Deviance:	    337600000 
Residual Deviance: 45140000 	AIC: 8484

$data
# A tibble: 602 × 5


In [13]:
compare_pangenomes(fit.lachnospirales, fit.oscillospirales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>   <dbl>               <dbl>
1 depth     11.8     102.      0.115 0.908                 -102.
2 istip   -175.       59.6    -2.94  0.00337               -267.
3 core   -1777.      762.     -2.33  0.0199               -3158.
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
              621.03               5995.25                343.43  
          istip:core       depth:pangenome       istip:pangenome  
              -31.35                 11.81               -175.23  
      core:pangenome  istip:core:pangenome  
            -1776.98               2220.95  

Degrees of Freedom: 1022 Total (i.e. Null);  1014 Residual
Null Deviance:	    658800000 
Residual Deviance: 83010000 	AIC: 14470

$data
# A tibble: 1,022

In [14]:
compare_pangenomes(fit.lachnospirales, fit.peptostreptococcales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>   <dbl>               <dbl>
1 depth   -169.      156.     -1.08    0.279               -520.
2 istip    -83.2      81.2    -1.02    0.306               -226.
3 core    -773.      917.     -0.843   0.399              -2415.
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
              621.03               5995.25                343.43  
          istip:core       depth:pangenome       istip:pangenome  
              -31.35               -168.85                -83.23  
      core:pangenome  istip:core:pangenome  
             -773.18               -559.04  

Degrees of Freedom: 882 Total (i.e. Null);  874 Residual
Null Deviance:	    592500000 
Residual Deviance: 84640000 	AIC: 12640

$data
# A tibble: 882 × 5

In [15]:
compare_pangenomes(fit.oscillospirales, fit.peptostreptococcales, family = "gaussian")

$summary
# A tibble: 3 × 7
  term  estimate std.error statistic p.value `bootstrap CI 2.5%`
  <chr>    <dbl>     <dbl>     <dbl>   <dbl>               <dbl>
1 depth   -181.      134.      -1.35   0.177              -557. 
2 istip     92.0      69.5      1.32   0.186               -87.0
3 core    1004.      743.       1.35   0.177              -309. 
# ℹ 1 more variable: `bootstrap CI 97.5%` <dbl>

$model

Call:  stats::glm(formula = model, family = family, data = dat)

Coefficients:
               istip                  core                 depth  
              445.80               4218.27                355.24  
          istip:core       depth:pangenome       istip:pangenome  
             2189.60               -180.65                 92.01  
      core:pangenome  istip:core:pangenome  
             1003.79              -2779.99  

Degrees of Freedom: 512 Total (i.e. Null);  504 Residual
Null Deviance:	    268500000 
Residual Deviance: 28310000 	AIC: 7062

$data
# A tibble: 512 × 5


#### Saving fits

In [ ]:
save.image(file = paste(task_root, "environment.RData", sep = "/"))

In [2]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Brussels
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] ggplot2_3.5.0   ape_5.7-1       panstripe_0.2.0

loaded via a namespace (and not attached):
 [1] crayon_1.5.2     vctrs_0.6.5      nlme_3.1-164     cli_3.6.2 